# yt-dlp server blocking test ? Firefox cookies

Use this notebook in Google Colab to test whether a Firefox-exported `cookies.txt` works for both channel listing and audio extraction.

Security note: `cookies.txt` is a secret. Upload it only into the temporary Colab runtime, never commit it, never paste it into logs, and revoke/refresh it if exposed.


In [ ]:
!python -m pip install -q -U "yt-dlp[default]" requests
!curl -fsSL https://deno.land/install.sh | sh >/dev/null


In [ ]:
import os
import platform
import shlex
import shutil
import subprocess
from pathlib import Path

import requests

os.environ["PATH"] = "/root/.deno/bin:" + os.environ.get("PATH", "")

CHANNEL_URL = "https://www.youtube.com/@zackdfilms/videos"
PLAYLIST_ENDS = [1, 3, 10, 25]
COOKIE_PATH = Path("cookies.txt")
DOWNLOAD_DIR = Path("/content/yt_dlp_cookie_downloads")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
FIREFOX_UA = "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:127.0) Gecko/20100101 Firefox/127.0"
COMMON_EJS_ARGS = ["--js-runtimes", "deno"]


def normalize_channel_url(url: str) -> str:
    url = url.strip()
    if "youtube.com/@" in url and not url.rstrip("/").endswith("/videos"):
        return url.rstrip("/") + "/videos"
    return url


def classify_output(output: str, returncode: int) -> str:
    lower = output.lower()
    bot_markers = [
        "sign in to confirm",
        "not a bot",
        "confirm you're not a bot",
        "confirm you?re not a bot",
        "unusual traffic",
    ]
    if any(marker in lower for marker in bot_markers):
        return "BLOCKED_OR_COOKIE_REJECTED"
    if "n challenge solving failed" in lower:
        return "EJS_CHALLENGE_SOLVER_FAILED"
    if "only images are available" in lower or "requested format is not available" in lower:
        return "NO_AUDIO_VIDEO_FORMATS"
    if any(marker in lower for marker in ["cookies", "cookiefile", "netscape", "http cookie file"]):
        if returncode != 0:
            return "COOKIE_FORMAT_OR_AUTH_ERROR"
    if "private video" in lower or "unavailable" in lower:
        return "VIDEO_UNAVAILABLE_OR_PRIVATE"
    if returncode == 0:
        return "SUCCESS"
    return f"FAILED_EXIT_{returncode}"


def run_cmd(label: str, args: list[str]) -> tuple[int, str, str]:
    print("=" * 90)
    print(label)
    print("$", " ".join(shlex.quote(str(arg)) for arg in args))
    proc = subprocess.run(args, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=os.environ.copy())
    output = proc.stdout or ""
    print(output)
    verdict = classify_output(output, proc.returncode)
    print("VERDICT:", verdict)
    return proc.returncode, output, verdict


def validate_cookie_file(path: Path) -> dict:
    if not path.exists():
        return {"ok": False, "reason": "missing_file", "youtube_cookie_count": 0, "first_line": None}
    text = path.read_text(encoding="utf-8", errors="ignore")
    lines = text.splitlines()
    first_line = lines[0].strip() if lines else ""
    valid_header = first_line.startswith("# Netscape HTTP Cookie File") or first_line.startswith("# HTTP Cookie File")
    cookie_lines = [line for line in lines if line and not line.startswith("#")]
    youtube_lines = [line for line in cookie_lines if "youtube.com" in line or ".youtube.com" in line or "google.com" in line]
    return {
        "ok": bool(valid_header and youtube_lines),
        "reason": "ok" if valid_header and youtube_lines else "invalid_header_or_no_youtube_cookies",
        "youtube_cookie_count": len(youtube_lines),
        "first_line": first_line,
    }


if shutil.which("ffmpeg") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=False)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=False)


In [ ]:
print("Python:", platform.python_version())
print("Platform:", platform.platform())
try:
    print("Public IP:", requests.get("https://api.ipify.org?format=json", timeout=10).json())
except Exception as exc:
    print("Public IP lookup failed:", exc)

run_cmd("yt-dlp version", ["yt-dlp", "--version"])
run_cmd("Deno version", ["deno", "--version"])
run_cmd("ffmpeg version", ["ffmpeg", "-version"])


## Upload `cookies.txt`

Export cookies from Firefox as Netscape-format `cookies.txt`, then upload it here. The notebook validates only file shape and cookie domains; it does not print cookie values.


In [ ]:
from google.colab import files

candidate_paths = [Path("cookies/cookies.txt"), Path("cookies.txt")]
existing = next((path for path in candidate_paths if path.exists()), None)
if existing is not None:
    if existing != COOKIE_PATH:
        COOKIE_PATH.write_text(existing.read_text(encoding="utf-8", errors="ignore"), encoding="utf-8")
    print(f"Using existing cookie file: {existing}")
else:
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No cookies file uploaded.")
    uploaded_name = next(iter(uploaded.keys()))
    uploaded_path = Path(uploaded_name)
    if uploaded_path != COOKIE_PATH:
        uploaded_path.replace(COOKIE_PATH)

cookie_status = validate_cookie_file(COOKIE_PATH)
print(cookie_status)
if not cookie_status["ok"]:
    raise RuntimeError(f"Invalid cookies.txt: {cookie_status['reason']}")


## Negative validation checks

These checks prove the validator catches missing and malformed cookie files before yt-dlp is called.


In [ ]:
print("Missing:", validate_cookie_file(Path("missing_cookies.txt")))
invalid_path = Path("invalid_cookies.txt")
invalid_path.write_text("not a netscape cookie file", encoding="utf-8")
print("Invalid:", validate_cookie_file(invalid_path))
invalid_path.unlink()


## Test 1 ? channel listing with cookies

This repeats the baseline listing test with `cookies.txt`, Firefox User-Agent, EJS/Deno, and conservative sleeps.


In [ ]:
channel_url = normalize_channel_url(CHANNEL_URL)
common_args = [
    *COMMON_EJS_ARGS,
    "--cookies", str(COOKIE_PATH),
    "--user-agent", FIREFOX_UA,
    "--sleep-interval", "3",
    "--max-sleep-interval", "6",
]

listing_results = []
for playlist_end in PLAYLIST_ENDS:
    listing_results.append(run_cmd(
        f"Channel flat playlist extraction with Firefox cookies, playlist-end={playlist_end}",
        [
            "yt-dlp",
            *common_args,
            "--flat-playlist",
            "--playlist-end", str(playlist_end),
            "--print", "%(id)s | %(title)s",
            channel_url,
        ],
    ))


## Test 2 ? one audio extraction with cookies

This validates the actual fallback path needed before Whisper transcription.


In [ ]:
audio_result = run_cmd(
    "One audio extraction with Firefox cookies",
    [
        "yt-dlp",
        *common_args,
        "--playlist-end", "1",
        "-x",
        "--audio-format", "mp3",
        "--audio-quality", "64K",
        "--paths", str(DOWNLOAD_DIR),
        "--output", "%(id)s.%(ext)s",
        channel_url,
    ],
)


In [ ]:
print("Summary")
print("- Cookie validation:", cookie_status)
for playlist_end, result in zip(PLAYLIST_ENDS, listing_results):
    print(f"- Listing {playlist_end}:", result[2])
print("- Audio:", audio_result[2])
print("Downloaded files:", [p.name for p in DOWNLOAD_DIR.glob("*")])

all_results = listing_results + [audio_result]
if all(result[2] == "SUCCESS" for result in all_results):
    print("Cookie workaround worked in this Colab runtime. Re-test on the target GPU/server provider before production integration.")
elif any(result[2] == "EJS_CHALLENGE_SOLVER_FAILED" for result in all_results):
    print("The blocker is EJS/challenge solving, not cookie expiry. Check Deno + yt-dlp[default] install output.")
elif any(result[2] == "BLOCKED_OR_COOKIE_REJECTED" for result in all_results):
    print("Cookies were rejected or YouTube still challenged the runtime. Refresh cookies or consider a non-yt-dlp fallback.")
else:
    print("Cookie workaround did not fully pass. Inspect the command output above before integrating.")
